# 04 — Sample characteristics, lesion coverage and deficit co-occurrence
Table 1, Supplementary figures S1 and S2.

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *
from mapstats import *
from figstyle import apply_figure_style, homogenize_fig
import matplotlib.pyplot as plt
from nilearn import plotting

brain, affine, BG = load_mask()
lab = impairment_labels()          # patients x domains impairment labels of the case-control samples
apply_figure_style()

## Table 1
Demographic and clinical rows describe all patients of the 12 cohorts; impairment rows are
n impaired / n available in the analysed case-control samples (impaired n = one-sample samples).

In [ ]:
clin = pd.read_csv(CLINICAL_CSV)
if "cognition-PSCI_present" in clin:                     # patients with harmonized cognitive assessment
    clin = clin[clin["cognition-PSCI_present"].notna()]
num = lambda c: pd.to_numeric(clin[c], errors="coerce")
vol = "stroke-total_infarct_volume" if "stroke-total_infarct_volume" in clin else "stroke-total_infarct_volume_ml"

def n_pct(mask, n=None):
    n = len(clin) if n is None else n
    return f"{int(mask.sum()):,} ({100 * mask.sum() / n:.1f})"

def med_iqr(s, d=0):
    s = s.dropna()
    return f"{s.median():.{d}f} [{s.quantile(.25):.{d}f}–{s.quantile(.75):.{d}f}]"

rows = [("Age, years, mean ± SD", f"{num('demographics-age').mean():.1f} ± {num('demographics-age').std():.1f}")]
for col, title in [("demographics-sex", "Sex"), ("demographics-education_STROKOG", "Education"),
                   ("demographics-ethnicity", "Ethnicity")]:
    rows.append((title, ""))
    rows += [(f"{lvl}, n (%)", n_pct(clin[col] == lvl)) for lvl in clin[col].value_counts().index]
rows += [("Baseline NIHSS, median [IQR]", med_iqr(num("stroke-NIHSS"))),
         ("Cognitive assessment timing, days after index stroke event, median [IQR]", med_iqr(num("stroke-interval_onset_NPO_days"))),
         ("Clinical history of stroke, n (%)", n_pct(num("stroke-prior_stroke") == 1)),
         ("Any-domain, n impaired / n available (%)",
          f"{int((lab == 1).any(axis=1).sum())} / {len(lab)} ({100 * (lab == 1).any(axis=1).mean():.1f})")]
for k in KEYS:
    n_imp, n = len(sample_ids(OS, k)), int(lab[k].notna().sum())
    rows.append((f"{DOMAINS[k]}, n impaired / n available (%)", f"{n_imp} / {n} ({100 * n_imp / n:.1f})"))
rows += [("Normalized acute infarct volume, mL, median [IQR]", med_iqr(num(vol), d=1)),
         ("Imaging timing, days after event, median [IQR]", med_iqr(num("stroke-interval_onset_MRI_days")))]
table1 = pd.DataFrame(rows, columns=["Characteristic", f"n = {len(clin):,}"])
table1.to_csv(TAB_DIR / "table1_sample_characteristics.csv", index=False)
table1

## Supplementary figure S1 — lesion coverage (analysed lesion masks)

In [ ]:
ids = sorted(set().union(*[sample_ids(VLSM, k) for k in KEYS]))
freq = sum((nib.load(f"{LESION_DIR}/{s}_infarct.nii.gz").get_fdata() > 0).astype(np.int32) for s in ids)
freq_img = nib.Nifti1Image(np.where(brain, freq, 0).astype(np.float32), affine)
fig, ax = plt.subplots(figsize=(8, 2.6))
d = plotting.plot_stat_map(freq_img, bg_img=BG, display_mode="z", cut_coords=[-18, -2, 14, 30, 46], axes=ax,
                           cmap="hot", colorbar=True, threshold=1, vmax=float(freq.max()), annotate=False,
                           black_bg=False, radiological=True, cbar_tick_format="%i")
a0 = list(d.axes.values())[0].ax
a0.text(0.03, 0.97, "R", transform=a0.transAxes, va="top", fontsize=9)
a0.text(0.97, 0.97, "L", transform=a0.transAxes, va="top", ha="right", fontsize=9)
homogenize_fig(fig); fig.savefig(FIG_DIR / "S01_lesion_coverage.png", dpi=300, bbox_inches="tight")
print(len(ids), "patients | max overlap", int(freq.max()))

## Supplementary figure S2 — co-occurrence of impairment labels (phi and Jaccard)

In [ ]:
phi = phi_matrix(lab)
jac = matrix(lambda a, b: 1.0 if a == b else ((lab[a] == 1) & (lab[b] == 1)).sum()
             / (lab[[a, b]].dropna() == 1).any(axis=1).sum(), KEYS)
phi.to_csv(TAB_DIR / "impairment_phi.csv"); jac.to_csv(TAB_DIR / "impairment_jaccard.csv")
print(f"mean phi = {offdiag(phi).mean():.2f} ± {offdiag(phi).std():.2f}, "
      f"mean Jaccard = {offdiag(jac).mean():.2f} ± {offdiag(jac).std():.2f}")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, M, title, cmap, vlim in [(axes[0], phi.values, "Impairment co-occurrence: phi (Pearson)", "RdBu_r", (-1, 1)),
                                 (axes[1], jac.values, "Impairment co-occurrence: Jaccard", "viridis", (0, offdiag(jac).max() * 1.3))]:
    im = ax.imshow(M, cmap=cmap, vmin=vlim[0], vmax=vlim[1])
    ax.set_xticks(range(6)); ax.set_xticklabels(SH.values(), rotation=45, ha="right")
    ax.set_yticks(range(6)); ax.set_yticklabels(SH.values())
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if (cmap == "viridis" and M[i, j] < vlim[1] * 0.5) else "black")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout(); homogenize_fig(fig)
fig.savefig(FIG_DIR / "S02_impairment_cooccurrence.png", dpi=300, bbox_inches="tight")